# 🔌 Open API 데이터 수집 완전 정복
## — 크롤링 없이 공식 데이터를 직접 받는 법

> **Open API란?**
> 웹사이트 화면을 긁지 않고, 제공자가 정해둔 주소로 요청하면
> 데이터를 깔끔하게 JSON/XML로 돌려주는 방식입니다.

---

## 학습 흐름

```
API 문서 확인
→ 인증키 발급
→ 요청 URL 구성
→ requests로 호출
→ JSON / XML 응답 확인
→ DataFrame 변환
→ CSV 저장
→ Tableau 시각화
```

## 이 노트북에서 배울 것

| 파트 | 내용 |
|------|------|
| 1 | API vs 크롤링 차이 + 우선순위 |
| 2 | API 요청 구조 (URL, params, 인증키) |
| 3 | JSON 응답 처리 → DataFrame |
| 4 | XML 응답 처리 |
| 5 | 공공데이터포털 사용 흐름 |
| 6 | 페이지 반복 수집 |
| 7 | 전처리 + CSV 저장 (Tableau용) |
| 8 | 실습 템플릿 + 포트폴리오 주제 |

---
## Part 1. API vs 크롤링 — 무엇이 다른가?

| 구분 | 웹 크롤링 | Open API |
|------|-----------|----------|
| 수집 대상 | 웹 페이지 HTML | 제공자가 정한 데이터 URL |
| 데이터 형식 | HTML → 파싱 필요 | JSON, XML (바로 사용 가능) |
| 안정성 | HTML 구조 변경에 취약 | 상대적으로 안정적 |
| 권한 | 사이트 정책 확인 필요 | 인증키 또는 활용신청 필요 |
| 실무 선호도 | API가 없을 때 사용 | **API가 있으면 우선 사용** |

### 수집 도구 선택 우선순위

```
공식 API가 있으면              → Open API 사용 (이 노트북)
API가 없고 HTML에 데이터 있으면 → BeautifulSoup (05 노트북)
JS 렌더링 동적 사이트이면       → Selenium + AI (06 노트북)
```

### 대표적인 Open API 제공처

| 기관 | 데이터 예시 |
|------|------------|
| 공공데이터포털 (data.go.kr) | 기상, 교통, 부동산, 문화행사 등 수천 종 |
| 기상청 (weather.go.kr) | 실시간 날씨, 예보, 미세먼지 |
| 카카오 개발자 | 지도, 검색, 로컬 |
| 네이버 개발자 | 검색, 쇼핑, 블로그 |
| 한국은행 (bok.or.kr) | 금리, 환율, 경제 통계 |

---
## Part 2. API 요청 구조 이해하기

API 요청은 보통 **URL + 파라미터** 조합으로 이루어집니다.

```
https://api.example.com/data?serviceKey=인증키&pageNo=1&numOfRows=10&type=json
│                             │              │          │              │
│                             인증키         페이지번호  한페이지데이터수  응답형식
└── 기본 요청 주소 (endpoint)
```

| 파라미터 | 의미 |
|----------|------|
| `serviceKey` | 나를 식별하는 인증키 (신청 후 발급) |
| `pageNo` | 몇 번째 페이지 |
| `numOfRows` | 한 페이지에 몇 개 데이터 |
| `type` | 응답 형식 (json / xml) |

### Python에서 params 딕셔너리를 쓰는 이유

```python
# ❌ URL에 직접 붙이는 방식 (한글 인코딩 오류 위험)
url = 'https://api.example.com/data?serviceKey=키&keyword=서울'

# ✅ params 딕셔너리 사용 (requests가 자동으로 인코딩)
params = {'serviceKey': '키', 'keyword': '서울'}
res = requests.get(url, params=params)
# → requests.get이 URL을 올바르게 인코딩해줌
```

In [ ]:
# API 기본 호출 구조 — 모든 API에 공통으로 쓰는 패턴

import requests
import pandas as pd
import time
import math

# ─── 요청 ────────────────────────────────────────────────────
base_url   = 'https://api.example.com/data'   # 실제 API URL로 교체
service_key = 'YOUR_API_KEY'                   # 발급받은 인증키로 교체

params = {
    'serviceKey': service_key,
    'pageNo':     1,
    'numOfRows':  10,
    'type':       'json'
}

res = requests.get(base_url, params=params, timeout=10)

# ─── 기본 확인 ───────────────────────────────────────────────
print(f"상태 코드: {res.status_code}")
print(f"실제 요청 URL: {res.url}")
print(f"응답 크기: {len(res.text):,} 글자")
print()
print("응답 원문 앞부분:")
print(res.text[:300])

---
## Part 3. JSON 응답 처리

### JSON이란?

Python의 딕셔너리/리스트와 거의 같은 구조입니다.

```json
{
  "response": {
    "body": {
      "totalCount": 100,
      "items": {
        "item": [
          {"name": "서울", "value": 10},
          {"name": "부산", "value": 20}
        ]
      }
    }
  }
}
```

실제 데이터는 보통 `response → body → items → item` 안에 있습니다.
(API마다 구조가 다르므로 API 문서 확인 필수)

### 응답 구조 탐색 방법

```python
data = res.json()

# 1단계: 최상위 키 확인
print(data.keys())

# 2단계: 한 단계씩 내려가기
print(data['response'].keys())
print(data['response']['body'].keys())

# 3단계: 실제 데이터 확인
items = data['response']['body']['items']['item']
print(type(items))   # list 이면 OK
print(items[0])      # 첫 번째 아이템 확인
```

In [ ]:
# JSON 응답 처리 실습 — 가짜 응답으로 연습
import json

# 실제 API 응답처럼 생긴 가짜 JSON
mock_response = {
    "response": {
        "header": {"resultCode": "00", "resultMsg": "NORMAL SERVICE."},
        "body": {
            "totalCount": 3,
            "pageNo": 1,
            "numOfRows": 10,
            "items": {
                "item": [
                    {"region": "서울", "pm10": 45, "pm25": 22, "date": "2024-01-15"},
                    {"region": "부산", "pm10": 38, "pm25": 18, "date": "2024-01-15"},
                    {"region": "대구", "pm10": 52, "pm25": 28, "date": "2024-01-15"}
                ]
            }
        }
    }
}

# ─── Step 1: 응답 구조 탐색 ──────────────────────────────────
print("=== Step 1: 최상위 키 확인 ===")
print(mock_response.keys())

print()
print("=== Step 2: response 안의 키 확인 ===")
print(mock_response['response'].keys())

print()
print("=== Step 3: body 안의 키 확인 ===")
body = mock_response['response']['body']
print(body.keys())
print(f"총 데이터 수: {body['totalCount']}")

print()
print("=== Step 4: 실제 데이터 확인 ===")
items = body['items']['item']
print(f"아이템 타입: {type(items)}")
print(f"첫 번째 아이템: {items[0]}")

In [ ]:
# ─── Step 5: DataFrame으로 변환 ──────────────────────────────

df = pd.DataFrame(items)

print("=== DataFrame 변환 완료 ===")
print(f"크기: {df.shape}")
print()
print(df)

print()
print("=== 컬럼 정보 ===")
print(df.dtypes)

In [ ]:
# API마다 응답 구조가 다릅니다. 자주 나오는 패턴들:

# 패턴 1: 최상위에 바로 리스트
case1 = [{"id": 1, "name": "A"}, {"id": 2, "name": "B"}]
df1 = pd.DataFrame(case1)
print("패턴 1 (최상위 리스트):", df1.shape)

# 패턴 2: items 키 안에 리스트
case2 = {"items": [{"id": 1, "name": "A"}, {"id": 2, "name": "B"}]}
df2 = pd.DataFrame(case2['items'])
print("패턴 2 (items 안 리스트):", df2.shape)

# 패턴 3: 공공데이터포털 표준 구조
case3 = {
    "response": {
        "body": {
            "totalCount": 2,
            "items": {
                "item": [{"sido": "서울", "cnt": 100}, {"sido": "부산", "cnt": 50}]
            }
        }
    }
}
df3 = pd.DataFrame(case3['response']['body']['items']['item'])
print("패턴 3 (공공데이터포털 표준):", df3.shape)

print()
print("─" * 40)
print("구조를 모를 때 탐색하는 법:")
print("""
data = res.json()

# 한 단계씩 들어가면서 확인
print(type(data))          # dict인지 list인지
print(data.keys())         # dict면 키 목록
print(data['키이름'])      # 그 안으로 들어가기
# → 실제 데이터(리스트)가 나올 때까지 반복
""")

---
## Part 4. XML 응답 처리

일부 API(특히 오래된 공공 API)는 XML 형식으로 응답합니다.

```xml
<response>
  <body>
    <items>
      <item>
        <region>서울</region>
        <pm10>45</pm10>
      </item>
      <item>
        <region>부산</region>
        <pm10>38</pm10>
      </item>
    </items>
  </body>
</response>
```

처리 방법: `BeautifulSoup`에 `'xml'` 파서를 사용합니다.

> ⚠️ XML 파서를 사용하려면 `lxml` 패키지가 필요합니다.
> 없으면 `pip install lxml` 실행 후 사용하세요.

In [ ]:
from bs4 import BeautifulSoup

# 가짜 XML 응답 (실제 API 응답처럼)
mock_xml = """
<response>
  <header>
    <resultCode>00</resultCode>
    <resultMsg>NORMAL SERVICE.</resultMsg>
  </header>
  <body>
    <totalCount>3</totalCount>
    <items>
      <item>
        <region>서울</region>
        <pm10>45</pm10>
        <pm25>22</pm25>
      </item>
      <item>
        <region>부산</region>
        <pm10>38</pm10>
        <pm25>18</pm25>
      </item>
      <item>
        <region>대구</region>
        <pm10>52</pm10>
        <pm25>28</pm25>
      </item>
    </items>
  </body>
</response>
"""

# XML 파싱 (lxml 파서)
soup_xml = BeautifulSoup(mock_xml, 'xml')

# item 태그 전부 찾기
items_xml = soup_xml.find_all('item')
print(f"아이템 수: {len(items_xml)}")

# 데이터 추출
rows_xml = []
for item in items_xml:
    region = item.find('region').text if item.find('region') else None
    pm10   = item.find('pm10').text   if item.find('pm10')   else None
    pm25   = item.find('pm25').text   if item.find('pm25')   else None
    
    rows_xml.append({'지역': region, 'PM10': pm10, 'PM2.5': pm25})

df_xml = pd.DataFrame(rows_xml)
print()
print(df_xml)

In [ ]:
# 실제 API에서 XML 응답 처리할 때의 코드 패턴

# res = requests.get(url, params=params, timeout=10)
# soup_xml = BeautifulSoup(res.text, 'xml')
# items_xml = soup_xml.find_all('item')

# rows = []
# for item in items_xml:
#     rows.append({
#         '필드1': item.find('태그명1').text if item.find('태그명1') else None,
#         '필드2': item.find('태그명2').text if item.find('태그명2') else None,
#     })

# df = pd.DataFrame(rows)

# ─── JSON vs XML 처리 비교 ─────────────────────────────────
print("=== JSON 처리 ===")
print("  data = res.json()")
print("  items = data['response']['body']['items']['item']")
print("  df = pd.DataFrame(items)")
print()
print("=== XML 처리 ===")
print("  soup = BeautifulSoup(res.text, 'xml')")
print("  items = soup.find_all('item')")
print("  for item in items: rows.append({...})")
print("  df = pd.DataFrame(rows)")
print()
print("→ JSON이 훨씬 편합니다. type=json 파라미터를 지원하면 JSON을 쓰세요!")

---
## Part 5. 공공데이터포털 사용 흐름

**사이트**: https://www.data.go.kr

### 인증키 발급 방법

```
1. data.go.kr 접속 → 회원가입/로그인
2. 원하는 데이터 검색 (예: '미세먼지', '전통시장', '문화행사')
3. 검색 결과에서 "오픈API" 탭 클릭 (파일데이터 아님!)
4. 원하는 API 클릭 → 상세 페이지
5. [활용신청] 버튼 클릭 → 활용목적 작성 (학습/연구 등)
6. 마이페이지 → 데이터활용 → 오픈API → 인증키 확인
7. 상세 페이지의 "참고문서" 다운로드 → 요청 URL과 파라미터 확인
```

### API 문서에서 확인해야 할 항목

```
□ 요청 URL (endpoint)
□ 인증키 파라미터 이름 (serviceKey? apiKey?)
□ 필수 파라미터 목록
□ 선택 파라미터 목록
□ 응답 형식 (JSON / XML, type 파라미터로 선택 가능한지)
□ 한 페이지 최대 데이터 수 (numOfRows 최대값)
□ 전체 건수가 응답에 포함되는지 (totalCount)
□ 일일 호출 제한 수
□ 데이터 갱신 주기
```

### 인증키 관리 원칙

```
□ GitHub 공개 저장소에 인증키 그대로 올리지 않기
□ 코드에는 'YOUR_API_KEY' 또는 '발급받은_인증키를_입력하세요' 로 표시
□ 포트폴리오 제출 전 인증키 부분 반드시 확인!
```

```python
# 안전한 방법: 변수로 분리
SERVICE_KEY = '발급받은_인증키를_입력하세요'

params = {
    'serviceKey': SERVICE_KEY,
    ...
}
```

---
## Part 6. 페이지 반복 수집

대부분의 API는 한번에 전체 데이터를 주지 않고 **페이지 단위**로 나눠서 줍니다.

```
pageNo=1 → 1~100번 데이터
pageNo=2 → 101~200번 데이터
pageNo=3 → 201~300번 데이터
...
```

### 방법 1: 고정 페이지 수 반복

```python
for page in range(1, 6):  # 5페이지까지
    ...
```

### 방법 2: totalCount 기반 자동 계산 (더 정확)

```python
# 1페이지 먼저 요청 → totalCount 확인
# → 전체 페이지 수 계산
# → 전체 페이지 반복
```

In [ ]:
# 방법 1: 고정 페이지 수 반복 수집 템플릿

base_url    = 'https://api.example.com/data'  # 실제 URL로 교체
service_key = 'YOUR_API_KEY'                   # 발급받은 키로 교체

all_items = []

for page in range(1, 6):  # 5페이지까지
    params = {
        'serviceKey': service_key,
        'pageNo':     page,
        'numOfRows':  100,
        'type':       'json'
    }
    
    res = requests.get(base_url, params=params, timeout=10)
    
    if res.status_code != 200:
        print(f"  {page}p 오류: {res.status_code}")
        break
    
    data = res.json()
    
    # 실제 API 구조에 맞게 수정
    items = data['response']['body']['items']['item']
    
    if not items:  # 빈 페이지면 종료
        print(f"  {page}p: 데이터 없음, 종료")
        break
    
    all_items.extend(items)
    print(f"  {page}p → {len(items)}개 수집 (누적: {len(all_items)}개)")
    
    time.sleep(0.5)  # 호출 간격 지키기!

df = pd.DataFrame(all_items)
print(f"\n총 {len(df)}개 수집 완료!")
# df.head()

In [ ]:
# 방법 2: totalCount 기반 전체 자동 수집 템플릿

base_url    = 'https://api.example.com/data'
service_key = 'YOUR_API_KEY'
NUM_OF_ROWS = 100  # 한 페이지당 데이터 수

# ─── Step 1: 첫 페이지 요청 → 전체 건수 확인 ───────────────
params = {
    'serviceKey': service_key,
    'pageNo':     1,
    'numOfRows':  NUM_OF_ROWS,
    'type':       'json'
}

res = requests.get(base_url, params=params, timeout=10)
data = res.json()

total_count = data['response']['body']['totalCount']  # 전체 데이터 수
total_pages = math.ceil(total_count / NUM_OF_ROWS)    # 올림으로 페이지 수 계산

print(f"전체 데이터: {total_count}개")
print(f"수집할 페이지: {total_pages}페이지")

# ─── Step 2: 전체 페이지 수집 ────────────────────────────────
all_items = []

# 첫 페이지 데이터 먼저 추가
first_items = data['response']['body']['items']['item']
all_items.extend(first_items)
print(f"  1p → {len(first_items)}개")

# 나머지 페이지
for page in range(2, total_pages + 1):
    params['pageNo'] = page
    
    res = requests.get(base_url, params=params, timeout=10)
    data = res.json()
    items = data['response']['body']['items']['item']
    
    all_items.extend(items)
    print(f"  {page}p → {len(items)}개 (누적: {len(all_items)}개)")
    
    time.sleep(0.5)

df_all = pd.DataFrame(all_items)
print(f"\n총 {len(df_all)}개 수집 완료!")

---
## Part 6-2. 자주 발생하는 오류와 해결법

| 문제 | 가능한 원인 | 해결 방법 |
|------|-------------|----------|
| 상태코드 400/401 | 인증키 누락 또는 잘못됨 | 인증키 재확인, URL 확인 |
| 상태코드 200인데 데이터 없음 | 필수 파라미터 누락 | API 문서의 필수 파라미터 확인 |
| 한글 검색 결과 없음 | 인코딩 문제 | `params` 딕셔너리 사용, URL 직접 조립 금지 |
| `KeyError` | 응답 구조 예상과 다름 | `res.text` 원문 출력 후 구조 확인 |
| 데이터 갑자기 없음 | 빈 페이지 도달 | `if not items: break` 처리 |
| 응답이 HTML | 인증키 오류 등으로 에러 페이지 반환 | `res.text[:300]` 출력 확인 |

In [ ]:
# 응답 오류 확인 — 항상 이 코드로 먼저 확인

def check_api_response(res):
    """API 응답 상태를 확인합니다."""
    print(f"상태 코드:   {res.status_code}")
    print(f"요청 URL:    {res.url}")
    print(f"응답 크기:   {len(res.text):,} 글자")
    print()
    
    # JSON인지 HTML인지 확인
    if res.text.strip().startswith('<'):
        print("⚠️  HTML 응답 (에러 페이지일 가능성!)")
    elif res.text.strip().startswith('{') or res.text.strip().startswith('['):
        print("✅ JSON 응답")
        try:
            data = res.json()
            # 공공데이터포털 표준 에러 확인
            if 'response' in data:
                header = data['response'].get('header', {})
                code = header.get('resultCode', 'N/A')
                msg  = header.get('resultMsg', 'N/A')
                print(f"   resultCode: {code}")
                print(f"   resultMsg:  {msg}")
                if code == '00':
                    print("   → 정상 응답!")
                else:
                    print("   → 오류 응답! 인증키/파라미터 확인 필요")
        except:
            print("   JSON 파싱 실패")
    
    print()
    print("응답 원문 앞부분 (300자):")
    print(res.text[:300])


print("check_api_response 함수 정의 완료!")
print()
print("사용법:")
print("  res = requests.get(url, params=params, timeout=10)")
print("  check_api_response(res)  ← 항상 이것부터!")

---
## Part 7. 데이터 전처리 + Tableau용 CSV 저장

In [ ]:
# API 데이터 전처리 표준 패턴
# (df에 실제 수집 데이터가 있다고 가정)

# 예시 DataFrame 생성
sample_data = {
    'region':  ['서울', '부산', '서울', '대구', None],
    'pm10':    ['45', '38', '45', '52', '60'],
    'pm25':    ['22', '18', '22', '28', None],
    'date':    ['20240115', '20240115', '20240115', '20240115', '20240115'],
    'comment': ['맑음', '맑음', '맑음', '약간나쁨', '나쁨']
}
df = pd.DataFrame(sample_data)
print("원본 데이터:")
print(df)
print()

# ─── 기본 확인 ───────────────────────────────────────────────
print("=== 기본 정보 ===")
print(f"크기: {df.shape}")
print(f"컬럼: {list(df.columns)}")
print()

print("=== 결측치 확인 ===")
print(df.isnull().sum())
print()

print("=== 데이터 타입 ===")
print(df.dtypes)

In [ ]:
# ─── 전처리 ──────────────────────────────────────────────────

df_clean = df.copy()

# 1. 중복 제거
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"중복 제거: {before} → {len(df_clean)}행")

# 2. 컬럼명 한글로 변경 (Tableau에서 보기 편하게)
df_clean = df_clean.rename(columns={
    'region':  '지역',
    'pm10':    'PM10',
    'pm25':    'PM2.5',
    'date':    '날짜',
    'comment': '상태'
})

# 3. 숫자형 변환 (API 응답은 문자열로 오는 경우 많음)
df_clean['PM10']  = pd.to_numeric(df_clean['PM10'],  errors='coerce')
df_clean['PM2.5'] = pd.to_numeric(df_clean['PM2.5'], errors='coerce')

# 4. 날짜형 변환 (yyyymmdd 형식 → datetime)
df_clean['날짜'] = pd.to_datetime(df_clean['날짜'], format='%Y%m%d', errors='coerce')

# 5. 결측치 처리 (필요한 경우만)
# df_clean = df_clean.dropna()           # 결측치 행 삭제
# df_clean = df_clean.fillna('데이터없음') # 문자열로 채우기

print()
print("=== 전처리 완료 ===")
print(df_clean.dtypes)
print()
print(df_clean)

In [ ]:
# Tableau 연결용 CSV 저장

import os

save_dir = os.path.join('data', 'results')
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, 'api_tableau_ready.csv')

df_clean.to_csv(save_path, index=False, encoding='utf-8-sig')

print(f"✅ 저장 완료: {save_path}")
print(f"   크기: {df_clean.shape[0]}행 × {df_clean.shape[1]}열")
print(f"   컬럼: {list(df_clean.columns)}")

print()
print("Tableau 연결 전 확인:")
checks = [
    ('컬럼명이 이해하기 쉬운가',       True),
    ('숫자 컬럼이 숫자형인가',         df_clean['PM10'].dtype in ['float64', 'int64']),
    ('날짜 컬럼이 datetime형인가',     str(df_clean['날짜'].dtype).startswith('datetime')),
    ('결측치가 너무 많지 않은가',      df_clean.isnull().sum().sum() < len(df_clean) * 0.3),
]
for desc, result in checks:
    print(f"  {'✅' if result else '⚠️ '} {desc}: {'OK' if result else '확인 필요'}")

---
## Part 8. 실습 템플릿 — 바로 쓸 수 있는 완성형 코드

공공데이터포털에서 인증키를 발급받으면 아래 코드의 `SERVICE_KEY`와 `BASE_URL`만 바꾸면 됩니다.

In [ ]:
import requests
import pandas as pd
import math
import time
import os

# ══════════════════════════════════════════════════════════════
#  여기만 수정하세요
# ══════════════════════════════════════════════════════════════
SERVICE_KEY  = '발급받은_인증키를_입력하세요'
BASE_URL     = 'API_요청_URL을_입력하세요'
NUM_OF_ROWS  = 100   # 한 페이지 최대 건수 (API 문서 확인)
SAVE_FILE    = 'api_result.csv'
# ══════════════════════════════════════════════════════════════

def fetch_page(page):
    """API에서 특정 페이지 데이터를 가져옵니다."""
    params = {
        'serviceKey': SERVICE_KEY,
        'pageNo':     page,
        'numOfRows':  NUM_OF_ROWS,
        'type':       'json'
        # 필요한 추가 파라미터는 여기에 입력
    }
    res = requests.get(BASE_URL, params=params, timeout=10)
    
    if res.status_code != 200:
        raise Exception(f"상태코드 오류: {res.status_code}")
    
    return res.json()


# ─── 1페이지로 전체 건수 확인 ────────────────────────────────
print("[1단계] 1페이지 요청 → 전체 건수 확인")
first_data  = fetch_page(1)
total_count = first_data['response']['body']['totalCount']
total_pages = math.ceil(total_count / NUM_OF_ROWS)
print(f"  전체 데이터: {total_count}개 | 페이지: {total_pages}p")

# ─── 전체 수집 ───────────────────────────────────────────────
print("\n[2단계] 전체 페이지 수집 시작")
all_items = []

# 첫 페이지 데이터 추가
all_items.extend(first_data['response']['body']['items']['item'])
print(f"  1p → {len(all_items)}개")

# 2페이지부터
for page in range(2, total_pages + 1):
    try:
        data  = fetch_page(page)
        items = data['response']['body']['items']['item']
        
        if not items:
            print(f"  {page}p: 빈 페이지, 종료")
            break
        
        all_items.extend(items)
        print(f"  {page}p → 누적 {len(all_items)}개")
        time.sleep(0.5)
    
    except Exception as e:
        print(f"  {page}p 오류: {e}")
        break

# ─── DataFrame + 저장 ────────────────────────────────────────
print("\n[3단계] DataFrame 변환 + 저장")
df_result = pd.DataFrame(all_items)

os.makedirs(os.path.join('data', 'results'), exist_ok=True)
save_path = os.path.join('data', 'results', SAVE_FILE)
df_result.to_csv(save_path, index=False, encoding='utf-8-sig')

print(f"✅ 완료: {len(df_result)}행 → {save_path}")
# df_result.head()

---
## 포트폴리오 주제 예시

| 주제 | API 데이터 | 분석 질문 |
|------|-----------|----------|
| 미세먼지 분석 | 기상청 대기질 API | 시간대별/지역별 차이는? 계절 영향은? |
| 지역 관광지 분석 | 공공데이터 관광정보 API | 어느 지역에 관광 콘텐츠가 많은가? |
| 문화 행사 분석 | 공연/전시 API | 월별 행사 수와 지역별 분포는? |
| 교통 사고 분석 | 교통사고 데이터 | 사고가 많은 지역과 시간대는? |
| 상권 분석 | 상권/매출 데이터 | 업종별, 지역별 매출 차이는? |
| 부동산 분석 | 국토부 실거래가 API | 지역별 가격 추이는? |

---

## README 템플릿 (GitHub 포트폴리오용)

```markdown
# Open API 데이터 수집 프로젝트

## 1. 프로젝트 개요
- 주제:
- 분석 목적:
- 사용 API:

## 2. 데이터 수집
- API 제공 기관:
- 수집 기간:
- 수집 건수:

## 3. 데이터 전처리
- 중복 제거:
- 결측치 처리:
- 형변환:

## 4. 주요 분석 질문
1.
2.
3.

## 5. Tableau 시각화
- 대시보드 링크:

## 6. 핵심 인사이트
1.
2.

## 7. 사용 기술
Python | requests | pandas | Open API | Tableau | GitHub
```

---
## ✅ 핵심 정리

### API 수집 5단계

```
Step 1: data.go.kr 에서 원하는 오픈API 검색 → 활용신청 → 인증키 발급

Step 2: API 문서 확인
         □ 요청 URL  □ 필수 파라미터  □ 응답 형식  □ totalCount 위치

Step 3: 1페이지 요청 → check_api_response(res) 로 응답 확인
         → JSON 구조 탐색 (data.keys() 한 단계씩)

Step 4: 전체 페이지 반복 수집 (totalCount 기반 자동 계산)
         → time.sleep(0.5) 호출 간격 지키기!

Step 5: 전처리 → CSV 저장 (utf-8-sig) → Tableau 연결
```

---

### 핵심 코드 3줄

```python
res  = requests.get(url, params=params, timeout=10)   # 요청
data = res.json()                                      # JSON 파싱
df   = pd.DataFrame(data['response']['body']['items']['item'])  # DataFrame
```

---

## 참고 자료

| 자료 | 링크 |
|------|------|
| 공공데이터포털 이용 가이드 | https://www.data.go.kr/ugs/selectPublicDataUseGuideView.do |
| Requests 공식 문서 | https://requests.readthedocs.io/en/latest/ |
| Python json 모듈 | https://docs.python.org/3/library/json.html |
| BeautifulSoup 공식 문서 (XML) | https://www.crummy.com/software/BeautifulSoup/bs4/doc/ |
| MDN HTTP 개요 | https://developer.mozilla.org/ko/docs/Web/HTTP |